In [0]:
from pyspark.sql.functions import *

silver_orders = spark.table("workspace.silver.silver_orders")
silver_customers = spark.table("workspace.silver.silver_customers")
silver_products = spark.table("workspace.silver.silver_products")
silver_order_items = spark.table("workspace.silver.silver_order_items")
silver_exchange_rates = spark.table("workspace.silver.silver_exchange_rates")

In [0]:
# dim_customers
dim_customers = silver_customers.select(
    "customer_id",
    "customer_name",
    "email",
    "country_code"
).dropDuplicates()

# dim_products
dim_products = silver_products.select(
    "product_id",
    "product_name",
    "category",
    "price",
    "country_code",
    "currency"
).dropDuplicates()

# dim_status
dim_status = silver_orders.select("order_status") \
    .distinct() \
    .withColumn("status_id", monotonically_increasing_id())

# dim_date
dim_date = silver_orders.select(
    to_date(col("order_date"), "dd-MM-yyyy").alias("date")
).dropDuplicates() \
.withColumn("year", year("date")) \
.withColumn("month", month("date")) \
.withColumn("day", dayofmonth("date"))

In [0]:
dim_customers.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_customers")
dim_products.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_products")
dim_status.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_status")
dim_date.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_date")